# FinAlign: prepare the passage index

Encode all **66,113 passages** once, then download the index for the
[FinAlign demo](https://huggingface.co/spaces/pdmdp/finalign).
The notebook uses the same BGE checkpoint, passage order and float32
embeddings as the demo. The completed ZIP includes the corpus and BM25
index; it can be reused across Space restarts.

1. Select **Runtime → Change runtime type → T4 GPU** and save.
2. Select **Runtime → Run all**. The environment installation runs first;
   passage encoding then prints progress approximately every 30 seconds.
3. Download **fiqa-index.zip**, then upload it without unpacking inside
   **Files → data/** in the Hugging Face Space.

Use a free Colab runtime. GPU access depends on availability and your
[Colab usage limits](https://research.google.com/colaboratory/faq.html).
No Hugging Face token, paid API or Google Drive mount is needed.
This run uses Colab compute, not the Space's ZeroGPU quota. It builds a
separate index; partial work already on the Space is not imported.
Keep the notebook open while it runs and download the result before
disconnecting. Rerunning cells resumes checkpoints while this runtime
still exists; Colab can delete its temporary files when it disconnects.


## 1. Install the encoder environment

Dependencies are installed in a separate environment to avoid changing
Colab's notebook packages. Only BGE is loaded; Qwen and monoT5 are used
later by the demo and are not needed to build document embeddings.


In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

if not shutil.which("nvidia-smi"):
    raise RuntimeError("Select Runtime > Change runtime type > T4 GPU, then reconnect and run again.")
gpu = subprocess.check_output(
    ["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"], text=True
).strip()
print("GPU:", gpu, flush=True)

ROOT = Path("/content/finalign-preparation")
ROOT.mkdir(parents=True, exist_ok=True)
ENV = ROOT / "venv"
PYTHON = ENV / "bin/python"
if not PYTHON.exists():
    bootstrap = ROOT / "bootstrap"
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "--no-cache-dir",
         "--target", str(bootstrap), "virtualenv==20.35.3"],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "virtualenv", "--no-download", str(ENV)],
        env={**os.environ, "PYTHONPATH": str(bootstrap)}, check=True,
    )

RUN_ENV = {
    **os.environ,
    "HF_HOME": str(ROOT / "model-cache"),
    "HF_HUB_DISABLE_IMPLICIT_TOKEN": "1",
    "TOKENIZERS_PARALLELISM": "false",
}
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--quiet", "--no-cache-dir",
     "torch==2.8.0", "--index-url", "https://download.pytorch.org/whl/cu126"],
    check=True, env=RUN_ENV,
)
subprocess.run(
    [str(PYTHON), "-m", "pip", "install", "--quiet", "--no-cache-dir",
     "sentence-transformers==6.3.0", "transformers==5.12.1",
     "huggingface-hub==1.21.0", "numpy==2.2.6", "pandas==2.2.3",
     "filelock==4.2.10", "sentencepiece==0.2.1"],
    check=True, env=RUN_ENV,
)
subprocess.run(
    [str(PYTHON), "-c", ("import torch; assert torch.cuda.is_available(), 'CUDA is unavailable'; "
     "print('PyTorch:', torch.__version__, '| GPU:', torch.cuda.get_device_name(0))")],
    check=True, env=RUN_ENV,
)
print("Environment ready.", flush=True)


## 2. Fetch the corpus and indexing code

The files come from a fixed revision of the public Space. Each download
is checked against its SHA-256 digest before it is used.


In [ ]:
import hashlib
from urllib.request import urlopen

SOURCE_REVISION = "c7d8739ff0ffbd5e9cb4fa73852c7425031b43dc"
BASE_URL = f"https://huggingface.co/spaces/pdmdp/finalign/resolve/{SOURCE_REVISION}"
FILES = {
    "finalign/__init__.py": "e10f80ad2147c5156ed0788b5284ce8685403a9a8b7f897f6500d086f8130ca4",
    "finalign/core.py": "85da32491835d00c722de08e1ef5c9d5ea40a3d79d0e048043f7755614aaf5f8",
    "finalign/index.py": "88a2412faddcb34f739dd97dddbbaf629e62565277d1b0b8bd59f8884ee4dde8",
    "finalign/preparation.py": "4b330c1b6a17555827e3b0e19eb37b8b2b5252b5848c423d699274f48363ce06",
    "data/fiqa-corpus.zip": "cdc3bfa6e82eddd9c3b7964b576552d73069270e1f058fa3811895e440d039e2",
}

def sha256(path):
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

for name, expected in FILES.items():
    destination = ROOT / name
    if not destination.exists() or sha256(destination) != expected:
        destination.parent.mkdir(parents=True, exist_ok=True)
        temporary = destination.with_suffix(destination.suffix + ".download")
        with urlopen(f"{BASE_URL}/{name}", timeout=120) as response, temporary.open("wb") as out:
            shutil.copyfileobj(response, out, length=1024 * 1024)
        if sha256(temporary) != expected:
            raise ValueError(f"Checksum mismatch: {name}. Run this cell again to retry the download.")
        temporary.replace(destination)
    print("Verified:", name, flush=True)


## 3. Encode and export

The worker uses the same `prepare_batch` and `finish_index` functions as
the Space. It records the encoder revision, dependency versions and
embedding checksum in the archive manifest, then reopens the ZIP to
verify its contents before offering it for download.


In [ ]:
BUILD_CODE = r"""
from pathlib import Path

import numpy as np
import torch
from sentence_transformers import SentenceTransformer

from finalign.index import Index, file_hash, unpack
from finalign.preparation import finish_index, prepare_batch

root = Path.cwd()
if not torch.cuda.is_available():
    raise RuntimeError("This notebook needs a CUDA GPU. Select a T4 runtime and reconnect.")
torch.set_float32_matmul_precision("highest")
index = Index(unpack(root / "data/fiqa-corpus.zip", root / "index"))
if len(index.passage_ids) != 66113 or len(index.documents) != 57600:
    raise ValueError("Unexpected corpus counts.")
print(f"Corpus: {len(index.documents):,} documents, {len(index.passage_ids):,} passages.", flush=True)

if not index.ready:
    specification = index.models["encoder"]
    encoder = SentenceTransformer(
        specification["id"], revision=specification["revision"], device="cuda"
    ).float().eval()
    encoder.max_seq_length = 512
    print("Encoder:", specification["id"], specification["revision"], flush=True)
    print("Encoding passages. Progress is saved and printed after each batch.", flush=True)
    while True:
        progress = prepare_batch(index, encoder, seconds=30, batch_size=32, device="cuda")
        completed, total = progress["completed"], progress["total"]
        elapsed = progress.get("seconds", 0.0)
        print(f"{completed:,} / {total:,} passages ({completed / total:.1%}) | "
              f"encoding time: {elapsed / 60:.1f} min", flush=True)
        if progress["done"]:
            break
    del encoder
    torch.cuda.empty_cache()

print("Packaging and checking the completed index...", flush=True)
archive = Path(finish_index(index, root / "fiqa-index.zip"))
verified = Index(unpack(archive, root / "verified-index"))
if not verified.ready or not np.isfinite(verified.embeddings).all():
    raise ValueError("The completed embedding matrix failed validation.")
print(f"Ready: {archive.name} ({archive.stat().st_size / 1_000_000:.1f} MB)", flush=True)
print("SHA-256:", file_hash(archive), flush=True)
print("Upload this ZIP, still compressed, inside the Space's data/ directory.", flush=True)
"""
subprocess.run(
    [str(PYTHON), "-u", "-c", BUILD_CODE],
    cwd=ROOT, env=RUN_ENV, check=True,
)


## 4. Download and upload to the Space

This downloads a ZIP of roughly 200–250 MB. You can also download it
from Colab's Files panel under `/content/finalign-preparation/`.

In the Space, open **Files → data/ → Add file → Upload files** and upload
**fiqa-index.zip**. Keep the file compressed. Commit the upload and leave
the hardware on **ZeroGPU**. After rebuilding, the app loads the saved
index and visitors can search without running document preparation.

ZeroGPU search still uses a visitor's GPU allowance and may queue or
reach a daily limit. Saving the index avoids repeating document encoding;
it does not remove those inference limits.


In [ ]:
from google.colab import files

archive = ROOT / "fiqa-index.zip"
if not archive.is_file():
    raise FileNotFoundError("Finish the encoding cell before downloading the index.")
files.download(str(archive))


Code: [FinAlign](https://github.com/pdmdp/FinAlign), MIT license.
Data and derived index: [FiQA / BEIR](https://huggingface.co/datasets/BeIR/fiqa),
CC BY-SA 4.0. Encoder: [BAAI/bge-base-en-v1.5](https://huggingface.co/BAAI/bge-base-en-v1.5).

Original project: Matteo Pietro Di Pilato, Lorenzo Cinquemani and
Lorenzo Marchioni. This notebook is part of the later interactive adaptation.
